# DeepLOB vs simple LOB predictors on Bybit order books

Downloads Bybit's free 200-level order book archive for a spot pair, rebuilds the book, samples it every 100 ms, and compares:
1. queue imbalance sign, 2. logistic regression on 9 handcrafted features (OFI, imbalance, spread, recent returns), 3. gradient boosting on the same, 4. DeepLOB (CNN + LSTM, Zhang, Zohren, Roberts 2019) on raw 100 x 40 windows.

Train day A, early-stop on day B, report on day C. The final table shows accuracy / macro-F1 and, more usefully, the realised mid-price change in bps after the model's confident up/down calls, so the number can be compared with the spread and the fee.

**Runtime: set to GPU** (Runtime > Change runtime type > T4 or better).

In [ ]:
SYMBOL = 'WIFUSDT'          # any Bybit spot pair, e.g. BTCUSDT, PEPEUSDT, ONDOUSDT
DAYS   = ['2026-09-01','2026-09-02','2026-09-03']   # train, validate, test (available since 2025-04-29)
H, K   = 10, 5               # horizon in 100 ms samples (10 = 1 s), smoothing window
EPOCHS, NTRAIN = 15, 400000  # DeepLOB epochs and training windows per epoch

import torch, os, subprocess
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none, training will be slow')

In [ ]:
import os, subprocess, sys, urllib.request, zipfile
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn'],check=False)
os.makedirs('ob', exist_ok=True)
for d in DAYS:
    f=f'ob/{d}_{SYMBOL}_ob200.data'
    if not os.path.exists(f):
        url=f'https://quote-saver.bycsi.com/orderbook/spot/{SYMBOL}/{d}_{SYMBOL}_ob200.data.zip'
        print('downloading', url); urllib.request.urlretrieve(url,'ob/tmp.zip')
        zipfile.ZipFile('ob/tmp.zip').extractall('ob'); os.remove('ob/tmp.zip')
print(os.listdir('ob'))

In [ ]:
# --- order book reconstruction -------------------------------------------
"""Rebuild Bybit orderbook.200 stream (snapshot + deltas) and sample top-L levels every DT ms.
Output .npz: t (ms), bp,bq,ap,aq arrays of shape (N, L)."""
import json, sys, numpy as np
def parse(path, L=10, dt_ms=100):
    bids={}; asks={}; out=[]; next_t=None
    with open(path) as f:
        for line in f:
            m=json.loads(line); d=m['data']; ts=m['ts']
            if m['type']=='snapshot':
                bids={float(p):float(q) for p,q in d['b']}; asks={float(p):float(q) for p,q in d['a']}
            else:
                for p,q in d['b']:
                    p=float(p); q=float(q)
                    if q==0: bids.pop(p,None)
                    else: bids[p]=q
                for p,q in d['a']:
                    p=float(p); q=float(q)
                    if q==0: asks.pop(p,None)
                    else: asks[p]=q
            if next_t is None: next_t=ts - ts%dt_ms + dt_ms
            while ts>=next_t:
                b=sorted(bids.items(),reverse=True)[:L]; a=sorted(asks.items())[:L]
                if len(b)==L and len(a)==L:
                    out.append((next_t, [x[0] for x in b],[x[1] for x in b],[x[0] for x in a],[x[1] for x in a]))
                next_t+=dt_ms
    t=np.array([o[0] for o in out]); bp=np.array([o[1] for o in out]); bq=np.array([o[2] for o in out])
    ap=np.array([o[3] for o in out]); aq=np.array([o[4] for o in out])
    return t,bp,bq,ap,aq
if False:
    src,dst=sys.argv[1],sys.argv[2]; L=int(sys.argv[3]) if len(sys.argv)>3 else 10; dt=int(sys.argv[4]) if len(sys.argv)>4 else 100
    t,bp,bq,ap,aq=parse(src,L,dt); np.savez_compressed(dst,t=t,bp=bp,bq=bq,ap=ap,aq=aq)
    mid=(bp[:,0]+ap[:,0])/2; spr=(ap[:,0]-bp[:,0])/mid*1e4
    print(f"{src}: {len(t):,} samples @ {dt}ms | median spread {np.median(spr):.2f} bps | spread==1 tick {np.mean((ap[:,0]-bp[:,0])<=1.0001*np.min(ap[:,0]-bp[:,0]))*100:.0f}% | mid moved between samples {np.mean(np.diff(mid)!=0)*100:.1f}%")


In [ ]:
for d in DAYS:
    src=f'ob/{d}_{SYMBOL}_ob200.data'; dst=f'ob/{d}.npz'
    if not os.path.exists(dst):
        t,bp,bq,ap,aq=parse(src,10,100); np.savez_compressed(dst,t=t,bp=bp,bq=bq,ap=ap,aq=aq)
        mid=(bp[:,0]+ap[:,0])/2; spr=(ap[:,0]-bp[:,0])/mid*1e4
        print(f'{d}: {len(t):,} samples | median spread {np.median(spr):.2f} bps | mid moved between samples {np.mean(np.diff(mid)!=0)*100:.1f}%')

In [ ]:
# --- features, labels, models, DeepLOB ------------------------------------
"""Compare LOB predictors on Bybit 10-level books sampled at 100 ms.
Label: direction of smoothed mid over horizon H samples (DeepLOB convention): up / flat / down with threshold alpha.
Models: (1) queue imbalance sign, (2) logistic on handcrafted features, (3) gradient boosting on same, (4) DeepLOB CNN-LSTM on raw 100x40 windows.
Train on day A, early-stop on day B, report on day C. Then convert to economics: mean realised mid change conditional on a confident up/down call."""
import sys, time, numpy as np
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu') if 'torch' in dir() else None
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score
import torch, torch.nn as nn
torch.manual_seed(0); np.random.seed(0)
def load(p):
    z=np.load(p); return z['bp'],z['bq'],z['ap'],z['aq']
def feats(bp,bq,ap,aq):
    mid=(bp[:,0]+ap[:,0])/2
    imb1=(bq[:,0]-aq[:,0])/(bq[:,0]+aq[:,0])
    imb3=(bq[:,:3].sum(1)-aq[:,:3].sum(1))/(bq[:,:3].sum(1)+aq[:,:3].sum(1))
    imb10=(bq.sum(1)-aq.sum(1))/(bq.sum(1)+aq.sum(1))
    spread=(ap[:,0]-bp[:,0])/mid
    # order flow imbalance (Cont et al.) at best level, summed over last 10 and 50 samples
    e=np.zeros(len(mid))
    for t in range(1,len(mid)):
        eb = bq[t,0] if bp[t,0]>bp[t-1,0] else (-bq[t-1,0] if bp[t,0]<bp[t-1,0] else bq[t,0]-bq[t-1,0])
        ea = aq[t,0] if ap[t,0]<ap[t-1,0] else (-aq[t-1,0] if ap[t,0]>ap[t-1,0] else aq[t,0]-aq[t-1,0])
        e[t]=eb-ea
    c=np.cumsum(e); ofi10=np.r_[np.zeros(10), c[10:]-c[:-10]]; ofi50=np.r_[np.zeros(50), c[50:]-c[:-50]]
    r10=np.r_[np.zeros(10), np.log(mid[10:]/mid[:-10])]; r50=np.r_[np.zeros(50), np.log(mid[50:]/mid[:-50])]
    depthratio=np.log((bq[:,:5].sum(1)+1e-9)/(aq[:,:5].sum(1)+1e-9))
    X=np.c_[imb1,imb3,imb10,spread,ofi10/(bq[:,0]+aq[:,0]+1e-9),ofi50/(bq[:,:3].sum(1)+aq[:,:3].sum(1)+1e-9),r10*1e4,r50*1e4,depthratio]
    return np.nan_to_num(X), mid, imb1
def labels(mid, H, K, alpha):
    # DeepLOB-style: compare mean of next H mids with mean of previous K mids
    cs=np.cumsum(np.r_[0.0,mid]); fut=(cs[K+H:]-cs[K:-H])/H if False else None
    m_prev=np.array([mid[max(0,t-K+1):t+1].mean() for t in range(len(mid))])
    m_next=np.full(len(mid),np.nan); s=np.cumsum(np.r_[0.0,mid]); m_next[:-H]=(s[H+1:]-s[1:-H])/H
    l=(m_next-m_prev)/m_prev; y=np.where(l>alpha,2,np.where(l<-alpha,0,1)); y[np.isnan(l)]=-1
    real=np.full(len(mid),np.nan); real[:-H]=(mid[H:]-mid[:-H])/mid[:-H]*1e4   # realised mid change in bps over H
    return y, real
def raw_windows(bp,bq,ap,aq,W=100):
    # DeepLOB input: (N, 1, W, 40) with levels [ap1,aq1,bp1,bq1,...], z-scored per day
    X=np.zeros((len(bp),40),dtype=np.float32)
    for i in range(10): X[:,4*i]=ap[:,i]; X[:,4*i+1]=aq[:,i]; X[:,4*i+2]=bp[:,i]; X[:,4*i+3]=bq[:,i]
    X=(X-X.mean(0))/(X.std(0)+1e-9); return X
class DeepLOB(nn.Module):
    def __init__(s,y_len=3):
        super().__init__()
        s.c1=nn.Sequential(nn.Conv2d(1,32,(1,2),(1,2)),nn.LeakyReLU(0.01),nn.Conv2d(32,32,(4,1),padding=(2,0)),nn.LeakyReLU(0.01),nn.Conv2d(32,32,(4,1),padding=(2,0)),nn.LeakyReLU(0.01))
        s.c2=nn.Sequential(nn.Conv2d(32,32,(1,2),(1,2)),nn.LeakyReLU(0.01),nn.Conv2d(32,32,(4,1),padding=(2,0)),nn.LeakyReLU(0.01),nn.Conv2d(32,32,(4,1),padding=(2,0)),nn.LeakyReLU(0.01))
        s.c3=nn.Sequential(nn.Conv2d(32,32,(1,10)),nn.LeakyReLU(0.01),nn.Conv2d(32,32,(4,1),padding=(2,0)),nn.LeakyReLU(0.01),nn.Conv2d(32,32,(4,1),padding=(2,0)),nn.LeakyReLU(0.01))
        s.i1=nn.Sequential(nn.Conv2d(32,64,(1,1)),nn.LeakyReLU(0.01),nn.Conv2d(64,64,(3,1),padding=(1,0)),nn.LeakyReLU(0.01))
        s.i2=nn.Sequential(nn.Conv2d(32,64,(1,1)),nn.LeakyReLU(0.01),nn.Conv2d(64,64,(5,1),padding=(2,0)),nn.LeakyReLU(0.01))
        s.i3=nn.Sequential(nn.MaxPool2d((3,1),stride=(1,1),padding=(1,0)),nn.Conv2d(32,64,(1,1)),nn.LeakyReLU(0.01))
        s.lstm=nn.LSTM(192,64,batch_first=True); s.fc=nn.Linear(64,y_len)
    def forward(s,x):
        x=s.c1(x); x=s.c2(x); x=s.c3(x)
        x=torch.cat([s.i1(x),s.i2(x),s.i3(x)],1)          # (B,192,T',1)
        x=x.squeeze(3).permute(0,2,1); o,_=s.lstm(x); return s.fc(o[:,-1])
def batches(X,y,idx,W,bs):
    for i in range(0,len(idx),bs):
        j=idx[i:i+bs]; xb=np.stack([X[t-W+1:t+1] for t in j])[:,None]; yield torch.from_numpy(xb), torch.from_numpy(y[j]).long()
def evaluate(name,p,y,real,mask):
    pred=p.argmax(1); acc=accuracy_score(y[mask],pred[mask]); f1=f1_score(y[mask],pred[mask],average='macro')
    conf=p.max(1); out=[]
    for q in [0.0,0.5,0.8]:
        thr=np.quantile(conf[mask],q); sel=mask&(conf>=thr)&(pred!=1)
        up=sel&(pred==2); dn=sel&(pred==0)
        edge=(np.nanmean(real[up]) if up.any() else 0) - (np.nanmean(real[dn]) if dn.any() else 0)
        hit=(np.mean(real[up]>0) if up.any() else np.nan, np.mean(real[dn]<0) if dn.any() else np.nan)
        out.append((q,sel.sum(),edge/2,hit))
    print(f"{name:22} acc {acc*100:5.1f}%  macroF1 {f1*100:5.1f}  | " + " | ".join(f"top{int((1-q)*100)}%: n={n:6d} edge {e:5.2f}bps hit {h[0]*100:4.0f}/{h[1]*100:4.0f}" for q,n,e,h in out))
def main(A,B,C,H=10,K=5,ALPHA=None,EPOCHS=15,NTRAIN=400000):
    W=100
    data={k:load(p) for k,p in zip('ABC',[A,B,C])}
    F={}; 
    for k,(bp,bq,ap,aq) in data.items():
        X,mid,imb=feats(bp,bq,ap,aq); tick=np.min(np.diff(np.unique(ap[:,0])))
        alpha=ALPHA if ALPHA is not None else 0.5*tick/np.median(mid)
        y,real=labels(mid,H,K,alpha); F[k]=(X,y,real,imb,raw_windows(bp,bq,ap,aq),mid)
    XA,yA,_,_,RA,_=F['A']; XB,yB,_,_,RB,_=F['B']; XC,yC,realC,imbC,RC,midC=F['C']
    mC=(yC>=0); mC[:W]=False
    print(f"H={H} samples ({H/10:.1f}s), K={K}, alpha={alpha*1e4:.2f} bps | test-day label mix down/flat/up: {[(yC[mC]==i).mean().round(3) for i in range(3)]}")
    # 1. imbalance sign
    p=np.zeros((len(yC),3)); p[:,2]=np.clip(imbC,0,1); p[:,0]=np.clip(-imbC,0,1); p[:,1]=1-np.abs(imbC); evaluate('queue imbalance sign',p,yC,realC,mC)
    # 2. logistic
    tr=yA>=0; lr=LogisticRegression(max_iter=500,C=1.0).fit(XA[tr],yA[tr]); evaluate('logistic, 9 features',lr.predict_proba(XC),yC,realC,mC)
    # 3. gradient boosting
    gb=HistGradientBoostingClassifier(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,early_stopping=True,validation_fraction=0.1,random_state=0).fit(XA[tr],yA[tr])
    evaluate('gradient boosting, 9 f',gb.predict_proba(XC),yC,realC,mC)
    # 4. DeepLOB
    DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); net=DeepLOB().to(DEV); opt=torch.optim.Adam(net.parameters(),lr=1e-3); lossf=nn.CrossEntropyLoss()
    idxA=np.where(yA>=0)[0]; idxA=idxA[idxA>=W]; idxB=np.where(yB>=0)[0]; idxB=idxB[idxB>=W]
    rng=np.random.default_rng(0); nA=NTRAIN; nB=40000
    subA=np.sort(rng.choice(idxA,min(nA,len(idxA)),replace=False)); subB=np.sort(rng.choice(idxB,min(nB,len(idxB)),replace=False))
    best=1e9; bad=0; t0=time.time()
    for ep in range(EPOCHS):
        net.train(); rng.shuffle(subA)
        for xb,yb in batches(RA,yA,subA,W,256):
            xb,yb=xb.to(DEV),yb.to(DEV); opt.zero_grad(); l=lossf(net(xb),yb); l.backward(); opt.step()
        net.eval(); vl=0; n=0
        with torch.no_grad():
            for xb,yb in batches(RB,yB,subB,W,1024): xb,yb=xb.to(DEV),yb.to(DEV); vl+=lossf(net(xb),yb).item()*len(yb); n+=len(yb)
        vl/=n; print(f"  DeepLOB epoch {ep+1}: val loss {vl:.4f} ({time.time()-t0:.0f}s)")
        if vl<best-1e-4: best=vl; bad=0; torch.save(net.state_dict(),'deeplob_best.pt')
        else:
            bad+=1
            if bad>=3: break
    net.load_state_dict(torch.load('deeplob_best.pt')); net.eval()
    idxC=np.where(mC)[0]; p=np.zeros((len(yC),3)); p[:,1]=1
    with torch.no_grad():
        for i in range(0,len(idxC),2048):
            j=idxC[i:i+2048]; xb=torch.from_numpy(np.stack([RC[t-W+1:t+1] for t in j])[:,None]); p[j]=torch.softmax(net(xb.to(DEV)),1).cpu().numpy()
    evaluate('DeepLOB (CNN-LSTM)',p,yC,realC,mC)
    print(f"\nreference: spread on test day median {np.median((data['C'][2][:,0]-data['C'][0][:,0])/midC*1e4):.2f} bps; Bybit spot fee base 10 bps maker / 10 bps taker")



In [ ]:
main(f'ob/{DAYS[0]}.npz', f'ob/{DAYS[1]}.npz', f'ob/{DAYS[2]}.npz', H=H, K=K, EPOCHS=EPOCHS, NTRAIN=NTRAIN)

## How to read the result

* `edge` is the average realised mid change (bps) after the model's up calls minus after its down calls, halved. Compare it with the pair's spread (printed) and with the fee you would pay: Bybit spot is 10 bps maker and taker at the base tier.
* `hit` is the share of up calls followed by an actual rise / down calls followed by an actual fall. On a large-tick pair most 1-second windows have no move at all, so hit rates look low for every model; the edge column is the fair comparison.
* If DeepLOB's edge is not clearly above the logistic row, the extra complexity is not buying anything, which is what the benchmark literature (LOBCAST, Briola et al.) reports.

To test the next-move question instead (when the mid next moves, which way?), see the `nextmove` cell below.

In [ ]:
# --- next-move direction: the classic large-tick test -----------------------
from sklearn.linear_model import LogisticRegression
z=np.load(f'ob/{DAYS[2]}.npz'); bp,bq,ap,aq=z['bp'],z['bq'],z['ap'],z['aq']; X,mid,imb=feats(bp,bq,ap,aq)
N=len(mid); nxt=np.zeros(N); last=0
for t in range(N-2,-1,-1):
    if mid[t+1]!=mid[t]: last=np.sign(mid[t+1]-mid[t])
    nxt[t]=last
ok=nxt!=0; y=(nxt>0).astype(int)
print(f'queue imbalance sign -> next move: acc {np.mean((imb[ok&(imb!=0)]>0)==(y[ok&(imb!=0)]==1))*100:.1f}%')
cut=int(N*0.6); tr=ok.copy(); tr[cut:]=False; te=ok.copy(); te[:cut]=False
lr=LogisticRegression(max_iter=500).fit(X[tr],y[tr]); p=lr.predict_proba(X[te])[:,1]; conf=np.abs(p-0.5); top=conf>=np.quantile(conf,0.8)
print(f'logistic -> next move: acc {np.mean((p>0.5)==(y[te]==1))*100:.1f}% overall, {np.mean((p[top]>0.5)==(y[te][top]==1))*100:.1f}% on the 20% most confident')